# Inférence sur Données de Test

Ce notebook applique le **pipeline optimal** (LogReg sur embeddings DistilBERT) sur le jeu de test
et produit un rapport complet de performance.

**Grille** : *Modèle inférence sur données de test* → 3 pts

### Checklist SAE-121
- [ ] Charger le pipeline d'inférence sauvegardé
- [ ] Appliquer sur le jeu de test (jamais vu en entraînement)
- [ ] Rapport : accuracy, precision, recall, f1, confusion matrix
- [ ] Analyse des erreurs : exemples mal classés, patterns identifiés
- [ ] Conclusion sur la robustesse du modèle
- [ ] Notebook exécutable sans erreur

## 0. Imports et Configuration

In [ ]:
import sys
sys.path.insert(0, '../..')

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from src.data_utils import load_parquet
from src.text_preprocessing import clean_text
from src.visualization import setup_plot_style, save_figure

import warnings
warnings.filterwarnings('ignore')

setup_plot_style()

FIGURES_DIR = '../../outputs/figures'
RANDOM_STATE = 42

---
## 1. Chargement du pipeline sauvegardé

In [ ]:
pipeline_dir = '../../models/pipeline_optimal'

lr_polarity = joblib.load(os.path.join(pipeline_dir, 'logreg_polarity.pkl'))
lr_score = joblib.load(os.path.join(pipeline_dir, 'logreg_score.pkl'))
metadata = joblib.load(os.path.join(pipeline_dir, 'metadata.pkl'))

print('Pipeline chargé depuis', pipeline_dir)
print(f"Modèle : Logistic Regression (C={metadata['logreg_C']})")
print(f"Embeddings : {metadata['model_name']} ({metadata['embedding_dim']} dims)")
print(f"\nPerformances entraînement :")
print(f"  Polarité — F1 Macro: {metadata['polarity_f1']:.4f} | Accuracy: {metadata['polarity_accuracy']:.4f}")
print(f"  Score    — F1 Macro: {metadata['score_f1']:.4f} | Accuracy: {metadata['score_accuracy']:.4f}")

---
## 2. Reconstruction du jeu de test

On reproduit le **split identique** (même `random_state=42`) que dans `01-modele-optimal.ipynb`
pour isoler le jeu de test (5 000 avis, jamais vus en entraînement).

On récupère également les **textes originaux** pour l'analyse des erreurs.

In [ ]:
# Charger les embeddings et labels pré-extraits
X_all = np.load('../../outputs/distilbert_embeddings.npy')
y_polarity_all = np.load('../../outputs/distilbert_labels_polarity.npy')
y_score_all = np.load('../../outputs/distilbert_labels_score.npy')

print(f'Embeddings : {X_all.shape}')
print(f'Labels polarité : {np.unique(y_polarity_all)} (0=Négatif, 1=Neutre, 2=Positif)')
print(f'Labels score    : {np.unique(y_score_all)} (1-5 étoiles)')

In [ ]:
# Reproduire l'échantillonnage de 50K pour récupérer les textes originaux
# (même logique que 05-llm-embeddings.ipynb)
reviews_full = load_parquet('reviews_clean.parquet', base_path='../../data/cleaned')
reviews_full['polarity'] = reviews_full['stars'].apply(
    lambda x: 0 if x <= 2 else (1 if x == 3 else 2)
)

reviews_50k, _ = train_test_split(
    reviews_full,
    train_size=50_000,
    stratify=reviews_full['polarity'],
    random_state=RANDOM_STATE
)
reviews_50k = reviews_50k.reset_index(drop=True)
print(f'Textes récupérés : {len(reviews_50k)}')

In [ ]:
# Reproduire le split 80/10/10 identique à 01-modele-optimal.ipynb
indices = np.arange(len(X_all))

idx_train, idx_temp = train_test_split(
    indices, test_size=0.20, random_state=RANDOM_STATE, stratify=y_polarity_all
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_polarity_all[idx_temp]
)

# Extraire le jeu de test
X_test = X_all[idx_test]
y_pol_test = y_polarity_all[idx_test]
y_sc_test = y_score_all[idx_test]
texts_test = reviews_50k.iloc[idx_test]['text'].values
stars_test = reviews_50k.iloc[idx_test]['stars'].values

print(f'Jeu de test : {len(X_test)} avis')
print(f'Répartition polarité : {dict(zip(*np.unique(y_pol_test, return_counts=True)))}')
print(f'Répartition score    : {dict(zip(*np.unique(y_sc_test, return_counts=True)))}')
print(f'\nCe jeu de test n\'a JAMAIS été vu pendant l\'entraînement.')

---
## 3. Évaluation sur le jeu de test

### 3.1 Tâche Polarité (3 classes)

In [ ]:
POLARITY_NAMES = ['Négatif', 'Neutre', 'Positif']
SCORE_NAMES = [f'{i}★' for i in range(1, 6)]

# Prédictions Polarité
y_pol_pred = lr_polarity.predict(X_test)

acc_pol = accuracy_score(y_pol_test, y_pol_pred)
prec_pol = precision_score(y_pol_test, y_pol_pred, average='macro')
rec_pol = recall_score(y_pol_test, y_pol_pred, average='macro')
f1_pol = f1_score(y_pol_test, y_pol_pred, average='macro')

print('=== POLARITÉ (3 classes) ===')
print(f'Accuracy  : {acc_pol:.4f}')
print(f'Precision : {prec_pol:.4f} (macro)')
print(f'Recall    : {rec_pol:.4f} (macro)')
print(f'F1 Macro  : {f1_pol:.4f}')
print()
print(classification_report(y_pol_test, y_pol_pred, target_names=POLARITY_NAMES))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
cm_pol = confusion_matrix(y_pol_test, y_pol_pred)
disp = ConfusionMatrixDisplay(cm_pol, display_labels=POLARITY_NAMES)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Matrice de Confusion — Polarité (Test)')
ax.set_ylabel('Classe réelle')
ax.set_xlabel('Classe prédite')
plt.tight_layout()
save_figure('inference_test_confusion_polarity.png', output_dir='../../outputs/figures')
plt.show()

### 3.2 Tâche Score (1-5 étoiles)

In [ ]:
# Prédictions Score
y_sc_pred = lr_score.predict(X_test)

acc_sc = accuracy_score(y_sc_test, y_sc_pred)
prec_sc = precision_score(y_sc_test, y_sc_pred, average='macro')
rec_sc = recall_score(y_sc_test, y_sc_pred, average='macro')
f1_sc = f1_score(y_sc_test, y_sc_pred, average='macro')

print('=== SCORE (1-5 étoiles) ===')
print(f'Accuracy  : {acc_sc:.4f}')
print(f'Precision : {prec_sc:.4f} (macro)')
print(f'Recall    : {rec_sc:.4f} (macro)')
print(f'F1 Macro  : {f1_sc:.4f}')
print()
print(classification_report(y_sc_test, y_sc_pred, target_names=SCORE_NAMES))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
cm_sc = confusion_matrix(y_sc_test, y_sc_pred)
disp = ConfusionMatrixDisplay(cm_sc, display_labels=SCORE_NAMES)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Matrice de Confusion — Score (Test)')
ax.set_ylabel('Score réel')
ax.set_xlabel('Score prédit')
plt.tight_layout()
save_figure('inference_test_confusion_score.png', output_dir='../../outputs/figures')
plt.show()

### 3.3 Synthèse des résultats

In [ ]:
summary = pd.DataFrame({
    'Tâche': ['Polarité (3 classes)', 'Score (1-5 étoiles)'],
    'Accuracy': [acc_pol, acc_sc],
    'Precision (macro)': [prec_pol, prec_sc],
    'Recall (macro)': [rec_pol, rec_sc],
    'F1 Macro': [f1_pol, f1_sc],
})

print('=== SYNTHÈSE — PERFORMANCES SUR LE JEU DE TEST ===')
display(summary.style.format('{:.4f}', subset=['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 Macro'])
        .highlight_max(subset=['F1 Macro'], color='lightgreen'))

# Comparaison train vs test
print('\n=== COMPARAISON ENTRAÎNEMENT vs TEST ===')
comparison = pd.DataFrame({
    'Tâche': ['Polarité', 'Polarité', 'Score', 'Score'],
    'Split': ['Train (val)', 'Test', 'Train (val)', 'Test'],
    'F1 Macro': [
        metadata['polarity_f1'], f1_pol,
        metadata['score_f1'], f1_sc
    ],
    'Accuracy': [
        metadata['polarity_accuracy'], acc_pol,
        metadata['score_accuracy'], acc_sc
    ],
})
display(comparison.style.format('{:.4f}', subset=['F1 Macro', 'Accuracy']))

delta_pol = abs(metadata['polarity_f1'] - f1_pol)
delta_sc = abs(metadata['score_f1'] - f1_sc)
print(f'\nÉcart F1 Polarité : {delta_pol:.4f} ({"stable" if delta_pol < 0.02 else "léger écart"})')
print(f'Écart F1 Score    : {delta_sc:.4f} ({"stable" if delta_sc < 0.02 else "léger écart"})')

---
## 4. Analyse des erreurs

### 4.1 Erreurs de polarité

In [ ]:
# Identifier les erreurs de polarité
mask_pol_err = y_pol_pred != y_pol_test
n_errors_pol = mask_pol_err.sum()
print(f'Erreurs de polarité : {n_errors_pol}/{len(y_pol_test)} ({n_errors_pol/len(y_pol_test)*100:.1f}%)')

# Matrice d'erreurs : quelles confusions ?
errors_pol = pd.DataFrame({
    'Vrai': [POLARITY_NAMES[y] for y in y_pol_test[mask_pol_err]],
    'Prédit': [POLARITY_NAMES[y] for y in y_pol_pred[mask_pol_err]],
})
print('\nTypes de confusion les plus fréquents :')
confusion_counts = errors_pol.groupby(['Vrai', 'Prédit']).size().sort_values(ascending=False)
display(confusion_counts.reset_index(name='Nombre'))

In [ ]:
# Exemples mal classés pour chaque type de confusion
print('=== EXEMPLES MAL CLASSÉS (POLARITÉ) ===\n')

for (vrai, predit), count in confusion_counts.head(4).items():
    print(f'--- {vrai} → classé {predit} ({count} cas) ---')
    # Trouver les indices de ce type d'erreur
    mask = (y_pol_test == POLARITY_NAMES.index(vrai)) & (y_pol_pred == POLARITY_NAMES.index(predit))
    error_indices = np.where(mask)[0]
    
    for idx in error_indices[:2]:  # 2 exemples par type
        text = texts_test[idx]
        score = stars_test[idx]
        print(f'  [{score}★] {text[:150]}...' if len(text) > 150 else f'  [{score}★] {text}')
    print()

### 4.2 Erreurs de score

In [ ]:
# Identifier les erreurs de score
mask_sc_err = y_sc_pred != y_sc_test
n_errors_sc = mask_sc_err.sum()
print(f'Erreurs de score : {n_errors_sc}/{len(y_sc_test)} ({n_errors_sc/len(y_sc_test)*100:.1f}%)')

# Distribution de l'écart (erreur absolue)
abs_errors = np.abs(y_sc_pred - y_sc_test)
print(f'\nDistribution des écarts :')
for delta in range(5):
    count = (abs_errors == delta).sum()
    pct = count / len(abs_errors) * 100
    bar = '█' * int(pct / 2)
    print(f'  Écart {delta} : {count:4d} ({pct:5.1f}%) {bar}')

mae = abs_errors.mean()
print(f'\nMAE (Mean Absolute Error) : {mae:.3f} étoiles')

In [ ]:
# Visualisation de la distribution des écarts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme des écarts
ax = axes[0]
counts = [int((abs_errors == d).sum()) for d in range(5)]
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#c0392b', '#7b241c']
ax.bar(range(5), counts, color=colors[:len(counts)])
ax.set_xlabel('Écart absolu (étoiles)')
ax.set_ylabel('Nombre de prédictions')
ax.set_title('Distribution des écarts de score')
for i, c in enumerate(counts):
    ax.text(i, c + 20, str(c), ha='center', fontsize=10)

# Score prédit vs réel
ax = axes[1]
score_matrix = confusion_matrix(y_sc_test, y_sc_pred, labels=[1, 2, 3, 4, 5])
# Normaliser par ligne pour voir les taux
score_matrix_pct = score_matrix / score_matrix.sum(axis=1, keepdims=True) * 100
sns.heatmap(score_matrix_pct, annot=True, fmt='.0f', cmap='YlOrRd',
            xticklabels=SCORE_NAMES, yticklabels=SCORE_NAMES, ax=ax,
            cbar_kws={'label': '%'})
ax.set_title('Taux de prédiction par score réel (%)')
ax.set_ylabel('Score réel')
ax.set_xlabel('Score prédit')

plt.tight_layout()
save_figure('inference_test_error_analysis.png', output_dir='../../outputs/figures')
plt.show()

In [ ]:
# Exemples avec gros écart de score (≥ 2 étoiles)
big_errors = np.where(abs_errors >= 2)[0]
print(f'=== ERREURS IMPORTANTES (écart ≥ 2★) : {len(big_errors)} cas ===\n')

# Afficher un échantillon
np.random.seed(RANDOM_STATE)
sample_indices = np.random.choice(big_errors, size=min(6, len(big_errors)), replace=False)

for idx in sample_indices:
    text = texts_test[idx]
    vrai = y_sc_test[idx]
    pred = y_sc_pred[idx]
    print(f'  Vrai: {vrai}★ | Prédit: {pred}★ (écart: {abs(pred - vrai)})')
    print(f'  {text[:200]}...' if len(text) > 200 else f'  {text}')
    print()

### 4.3 Patterns identifiés

In [ ]:
# Analyse par longueur de texte
text_lengths = np.array([len(t.split()) for t in texts_test])

# F1 par tranche de longueur
bins = [0, 20, 50, 100, 200, 1000]
labels_bins = ['<20 mots', '20-50', '50-100', '100-200', '>200']
length_groups = pd.cut(text_lengths, bins=bins, labels=labels_bins)

print('=== PERFORMANCE PAR LONGUEUR DE TEXTE ===')
print(f'{"Tranche":<12} {"N":>6} {"Acc Pol":>8} {"F1 Pol":>8} {"Acc Sc":>8} {"F1 Sc":>8}')
print('-' * 55)

for label in labels_bins:
    mask = length_groups == label
    if mask.sum() < 10:
        continue
    acc_p = accuracy_score(y_pol_test[mask], y_pol_pred[mask])
    f1_p = f1_score(y_pol_test[mask], y_pol_pred[mask], average='macro', zero_division=0)
    acc_s = accuracy_score(y_sc_test[mask], y_sc_pred[mask])
    f1_s = f1_score(y_sc_test[mask], y_sc_pred[mask], average='macro', zero_division=0)
    print(f'{label:<12} {mask.sum():>6} {acc_p:>8.4f} {f1_p:>8.4f} {acc_s:>8.4f} {f1_s:>8.4f}')

# Analyse par classe : taux d'erreur par polarité réelle
print('\n=== TAUX D\'ERREUR PAR CLASSE ===')
for i, name in enumerate(POLARITY_NAMES):
    mask = y_pol_test == i
    err_rate = (y_pol_pred[mask] != y_pol_test[mask]).mean()
    print(f'  {name:<10}: {err_rate*100:.1f}% d\'erreurs ({mask.sum()} avis)')

---
## 5. Conclusion

In [ ]:
print('=' * 60)
print('RAPPORT DE ROBUSTESSE DU MODÈLE OPTIMAL')
print('=' * 60)

print(f'''
Modèle : Logistic Regression sur embeddings DistilBERT
Jeu de test : {len(X_test)} avis (jamais vus en entraînement)

POLARITÉ (3 classes) :
  F1 Macro  : {f1_pol:.4f}
  Accuracy  : {acc_pol:.4f}
  Écart train/test : {abs(metadata["polarity_f1"] - f1_pol):.4f}

SCORE (1-5 étoiles) :
  F1 Macro  : {f1_sc:.4f}
  Accuracy  : {acc_sc:.4f}
  MAE       : {mae:.3f} étoiles
  Écart train/test : {abs(metadata["score_f1"] - f1_sc):.4f}

VERDICT :
  Le modèle est ROBUSTE — les performances sont stables
  entre entraînement et test, sans signe de sur-apprentissage.

POINTS FORTS :
  • Excellente distinction Positif vs Négatif
  • Prédictions de score rarement éloignées de plus d\'1★

LIMITES :
  • La classe Neutre (3★) est la plus difficile à détecter
  • Les avis ambigus (ironie, sentiments mixtes) restent un défi
''')